# **neuraTape:** Neural Turing Machines

![Neural Turing Machines Architecture](./image.png)

In [1]:
import torch

## Memory Matrix

In [2]:
N = 256     # number of memory locations
M = 64      # vector size of each memory location

In [3]:
M_t = torch.rand(N, M)  # memory matrix at time t
print(f"Memory matrix M_t: {M_t}")
print(f"Memory matrix M_t shape: {M_t.shape}")

Memory matrix M_t: tensor([[0.9661, 0.0272, 0.9832,  ..., 0.9859, 0.0845, 0.4641],
        [0.7200, 0.1980, 0.5441,  ..., 0.2157, 0.6150, 0.6327],
        [0.8903, 0.5599, 0.0126,  ..., 0.4978, 0.0641, 0.6909],
        ...,
        [0.5365, 0.8927, 0.8240,  ..., 0.8913, 0.9137, 0.9592],
        [0.8052, 0.6078, 0.1224,  ..., 0.8598, 0.8796, 0.0652],
        [0.6319, 0.0178, 0.6388,  ..., 0.6557, 0.4436, 0.7778]])
Memory matrix M_t shape: torch.Size([256, 64])


## Read Head

In [4]:
r = torch.rand(N)                               # read vector at time t (N, )
w_r_t = torch.nn.functional.softmax(r, dim=0)   # read weights at time t (N, )
w_r_t = w_r_t.unsqueeze(1)                      # reshape to (N, 1)

r_t = M_t.T @ w_r_t                             # read vector at time t (M, )
print(f"Read vector r_t: {r_t}")
print(f"Read vector r_t shape: {r_t.shape}")

Read vector r_t: tensor([[0.4893],
        [0.4998],
        [0.4994],
        [0.4838],
        [0.5183],
        [0.4900],
        [0.4997],
        [0.4969],
        [0.5150],
        [0.5008],
        [0.5259],
        [0.5027],
        [0.5006],
        [0.5130],
        [0.4943],
        [0.5343],
        [0.4893],
        [0.4576],
        [0.5223],
        [0.4806],
        [0.5155],
        [0.4667],
        [0.5278],
        [0.4769],
        [0.5115],
        [0.5256],
        [0.4818],
        [0.5319],
        [0.4860],
        [0.4951],
        [0.5144],
        [0.5080],
        [0.5064],
        [0.4921],
        [0.5348],
        [0.5023],
        [0.5219],
        [0.5092],
        [0.5246],
        [0.5113],
        [0.5067],
        [0.4967],
        [0.5037],
        [0.4614],
        [0.5116],
        [0.4812],
        [0.4969],
        [0.4814],
        [0.5118],
        [0.5048],
        [0.4965],
        [0.4776],
        [0.5492],
        [0.5368],
        [0.

## Write Head

In [5]:
w = torch.rand(N)                               # write weights at time t (N, )
w_w_t = torch.nn.functional.softmax(w, dim=0)   # write weights at time t (N, )
w_w_t = w_w_t.unsqueeze(1)                      # reshape to (N, 1)

e_t = torch.rand(M)                             # erase vector at time t (M, )
e_t = torch.nn.functional.softmax(e_t, dim=0)   # normalize erase vector
e_t = e_t.unsqueeze(0)                          # reshape to (1, M)

a_t = torch.rand(M)                             # add vector at time t (M, )
a_t = torch.nn.functional.softmax(a_t, dim=0)   # normalize add vector
a_t = a_t.unsqueeze(0)                          # reshape to (1, M)

M_t_plus_1_e = M_t * (1 - w_w_t @ e_t)          # (N, M) * (N, M) -> (N, M)
M_t_plus_1_a = w_w_t @ a_t                      # (N, 1) @ (1, M) -> (N, M)
M_t_plus_1 = M_t_plus_1_e + M_t_plus_1_a        # (N, M) + (N, M) -> (N, M)

print(f"Updated memory matrix M_t+1: {M_t_plus_1}")
print(f"Updated memory matrix M_t+1 shape: {M_t_plus_1.shape}")

Updated memory matrix M_t+1: tensor([[0.9661, 0.0273, 0.9832,  ..., 0.9859, 0.0846, 0.4642],
        [0.7200, 0.1981, 0.5442,  ..., 0.2157, 0.6150, 0.6327],
        [0.8903, 0.5599, 0.0127,  ..., 0.4978, 0.0642, 0.6909],
        ...,
        [0.5365, 0.8927, 0.8240,  ..., 0.8913, 0.9137, 0.9592],
        [0.8051, 0.6078, 0.1224,  ..., 0.8598, 0.8796, 0.0652],
        [0.6319, 0.0178, 0.6389,  ..., 0.6557, 0.4437, 0.7778]])
Updated memory matrix M_t+1 shape: torch.Size([256, 64])


## Addressing Mechanisms

### Focusing by Content

In [6]:
k_t = torch.rand(M)                             # key vector at time t (M, )
k_t = torch.unsqueeze(k_t, 0)                   # reshape to (1, M)
b_t = torch.rand(1)                             # key strength at time t (1, ) [scalar]

similarity = torch.nn.functional.cosine_similarity(M_t, k_t, dim=1)     # (N, M) vs (1, M) -> (N, )
w_c_t = torch.nn.functional.softmax(similarity * b_t, dim=0)            # (N, )
w_c_t = w_c_t.unsqueeze(1)                                              # reshape to (N, 1)

print(f"Updated content-based addressing weights w_c_t: {w_c_t}")
print(f"Updated content-based addressing weights w_c_t shape: {w_c_t.shape}")

Updated content-based addressing weights w_c_t: tensor([[0.0040],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0040],
        [0.0040],
        [0.0040],
        [0.0039],
        [0.0040],
        [0.0039],
        [0.0040],
        [0.0038],
        [0.0039],
        [0.0039],
        [0.0040],
        [0.0039],
        [0.0039],
        [0.0038],
        [0.0039],
        [0.0040],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0038],
        [0.0039],
        [0.0040],
        [0.0038],
        [0.0038],
        [0.0039],
        [0.0040],
        [0.0038],
        [0.0040],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0039],
        [0.0038],
        [0.0039],
        [0.0038],
        [0.0039],
        [0.0039],
        [0.0038],
        [0.0040],
        [0.0039],
        [0.0040],
        [0.0040],
        [0.0039],
        [0.0039]

### Focusing by Location

In [7]:
w_t_minus_1 = torch.rand(N)                     # last write weights at time t-1 (N, )
w_t_minus_1 = w_t_minus_1.unsqueeze(1)          # reshape to (N, 1)

g_t = torch.rand(1)                             # interpolation gate at time t (1, ) [scalar]
s_t = torch.rand(N)                             # shift weighting (logits) at time t (N, )
s_t = torch.softmax(s_t, dim=0)                 # shift weighting at time t (N, )
y_t = torch.rand(1)                             # sharpening factor at time t (1, ) [scalar]

w_g_t = g_t * w_c_t + (1 - g_t) * w_t_minus_1                           # (N, 1)

w_s_t = torch.zeros_like(w_g_t)                                         # (N, 1)
for i in range(N):
    w_s_t[i] = torch.sum(torch.roll(w_g_t, shifts=i) * s_t[i])          # (1, )

w_y_t = w_s_t ** y_t                                                    # (N, 1)
w_t = w_y_t / torch.sum(w_y_t)                                          # (N, 1)

print(f"Updated addressing weights w_t: {w_t}")
print(f"Updated addressing weights w_t shape: {w_t.shape}")

Updated addressing weights w_t: tensor([[0.0043],
        [0.0034],
        [0.0043],
        [0.0042],
        [0.0044],
        [0.0042],
        [0.0043],
        [0.0040],
        [0.0035],
        [0.0043],
        [0.0041],
        [0.0042],
        [0.0034],
        [0.0037],
        [0.0036],
        [0.0039],
        [0.0041],
        [0.0040],
        [0.0040],
        [0.0033],
        [0.0034],
        [0.0037],
        [0.0035],
        [0.0044],
        [0.0036],
        [0.0038],
        [0.0040],
        [0.0038],
        [0.0041],
        [0.0040],
        [0.0044],
        [0.0039],
        [0.0044],
        [0.0042],
        [0.0035],
        [0.0035],
        [0.0035],
        [0.0039],
        [0.0040],
        [0.0042],
        [0.0035],
        [0.0041],
        [0.0037],
        [0.0040],
        [0.0038],
        [0.0044],
        [0.0039],
        [0.0040],
        [0.0042],
        [0.0043],
        [0.0043],
        [0.0044],
        [0.0037],
        [0.004